# RS01 · Testable firmware - HAL, dependency injection, test doubles, coverage

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/07_reliability_systems/RS01_testable_firmware_hal_dependency_injection_test_.ipynb)

**Track:** Reliability and systems · **Skill axis (sheet):** Reliability and systems · **Time:** about 65 min

**Skill Hub concepts:** Hardware abstraction layer and dependency injection; Test doubles and coverage

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("RS01")

## Map: cut the module off from the hardware so a PC can test it

```
 production:  motor.c ──calls──> MotorHal { pwm_set, adc_read_mA, millis, ctx } ──> STM32 HAL / registers
 host test:   motor.c ──calls──> MotorHal { fake_pwm_set, fake_adc, fake_millis, &fake_state }   (gcc on a PC)
```

| Double | What it does | Example here |
|---|---|---|
| Stub | returns canned values | `adc_read_mA` always returns 500 |
| Fake | working lightweight implementation | a "PWM" that stores the duty in a struct you can inspect |
| Spy | records how it was called | counts `pwm_set` calls, keeps the last argument |
| Mock | expects specific calls, fails the test otherwise | "pwm_set(0) must be called once on fault" |

Injection styles in C: **function-pointer struct** (runtime, shown here), **link-time substitution** (link `fake_hal.c`
instead of `stm32_hal.c`), **preprocessor** (`#ifdef TEST`, least clean).

**Coverage** (`gcc --coverage`, then `gcov`): line coverage = executed lines / executable lines; branch coverage
counts each outcome of each decision. High coverage shows what tests never touched; it does not prove correctness.

Module under test, `motor_step(m, target)` each 1 ms tick:
1. If faulted: return -1 (PWM stays 0).
2. If `adc_read_mA()` > 3000: fault: `pwm_set(0)`, return -1.
3. Otherwise move the duty toward `target` by at most 100 per call (soft start), `pwm_set(duty)`, return the duty.

In [ ]:
%%writefile motor.h
#include <stdint.h>
typedef struct {
    void (*pwm_set)(void *ctx, uint16_t duty);
    uint16_t (*adc_read_mA)(void *ctx);
    void *ctx;
} MotorHal;
typedef struct { const MotorHal *hal; uint16_t duty; int faulted; } Motor;
void motor_init(Motor *m, const MotorHal *hal);
int motor_step(Motor *m, uint16_t target);

## Task 1: the module, using only the injected HAL

In [ ]:
%%writefile motor.c
#include "motor.h"

void motor_init(Motor *m, const MotorHal *hal) {
    m->hal = hal;
    m->duty = 0;
    m->faulted = 0;
    hal->pwm_set(hal->ctx, 0);
}

int motor_step(Motor *m, uint16_t target) {
    (void)m; (void)target; return 0; /* TODO */
}

## Task 2: your own fake HAL and tests
Write a fake that stores the last duty and counts calls, and an ADC stub you can set. Test soft start, ramp-down,
the over-current fault and that a faulted motor stays off. The checker runs your tests with coverage and wants every
test to pass and at least 90% line coverage of `motor.c`.

In [ ]:
%%writefile test_motor.c
#include <stdio.h>
#include "motor.h"

typedef struct { uint16_t last_duty; int pwm_calls; uint16_t adc_mA; } Fake;
static void fake_pwm(void *ctx, uint16_t d) { Fake *f = ctx; f->last_duty = d; f->pwm_calls++; }
static uint16_t fake_adc(void *ctx) { return ((Fake *)ctx)->adc_mA; }

static int fails = 0;
#define EXPECT(c) do { if (!(c)) { printf("FAIL line %d: %s\n", __LINE__, #c); fails++; } } while (0)

int main(void) {
    Fake f = {0, 0, 500};
    MotorHal hal = {fake_pwm, fake_adc, &f};
    Motor m;
    motor_init(&m, &hal);
    /* TODO: tests with EXPECT(...) */
    printf(fails ? "TESTS FAILED\n" : "ALL TESTS PASSED\n");
    return fails != 0;
}

In [ ]:
from roboprep import clab
print(clab.sh("rm -f *.gcda; gcc --coverage -O0 -c motor.c test_motor.c && gcc --coverage motor.o test_motor.o -o t_motor "
             "&& ./t_motor && gcov -n motor.c | head -3").stdout)
lab.check("module")
lab.check("tests")

## Checkpoint: Math
**M1 (Apply).** gcov reports 19 of 20 executable lines run. Line coverage in percent?

**M2 (Apply).** Motor at duty 0, target 1000. How many 1 ms ticks until the duty reaches 1000?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Understand).** Your test double records every `pwm_set` argument so the test can assert the ramp shape afterwards.
Which kind is it?
- a) stub
- b) spy (or a fake with spying)
- c) mock

**K2 (Evaluate).** Compare function-pointer injection with link-time substitution for an STM32 project: code size,
speed, flexibility in tests, and what each costs in readability.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `rs01.h`).
3. In `roboprep/checks/rs01.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()